In [ ]:


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import numpy as np
#from sklearn.decomposition import PCA
#from sklearn.neighbors import NearestNeighbors
#import igraph as ig
#import leidenalg as la
#from umap.umap_ import fuzzy_simplicial_set
#import umap
import os
import math
#import pickle
from natsort import natsorted
from pathlib import Path

import anndata as ad
import warnings
warnings.filterwarnings('ignore')
#import tensorflow as tf


import sys
import pickle


In [ ]:
home_dir = Path.home()

src_dir = home_dir / 'ext_hd_sammy' / 'projects' / 'out' / 'out_msi' 
dst_folder = home_dir / 'ext_hd_sammy' / 'projects' / 'out' / 'out_msi' / 'script05_output'
os.makedirs(dst_folder, exist_ok=True)

In [ ]:
peptides = src_dir / 'peptide_msi'
glycans = src_dir / 'glycan_msi'
metab = src_dir / 'metab_msi'

os.listdir(glycans)

In [ ]:
import os
import pandas as pd
import tqdm

# Define data path
data = os.listdir(glycans) # List of file names
data_path = glycans / data[2]

# Initialize the first dataframe
obs = None

files = os.listdir(data_path)
# Iterate through all glycan files
for i in tqdm.tqdm(range(len(files))):
    file_path = os.path.join(data_path, files[i])

    # Read file with optimized dtypes
    df_temp = pd.read_parquet(file_path)

    # Extract proper channel name from the filename
    channel_name = '_'.join(files[i].split('.')[0].split('_')[1:4])  

    # Select only necessary columns and rename intensity column
    df_temp = df_temp[['x', 'y', 'intensity']].copy()
    df_temp.rename(columns={'intensity': channel_name}, inplace=True)

    # Convert to smaller dtype to reduce memory
    df_temp[channel_name] = df_temp[channel_name].astype('float32')

    # Merge iteratively to reduce memory overhead
    if obs is None:
        obs = df_temp
    else:
        obs = obs.merge(df_temp, on=['x', 'y'], how='outer')

    # Free up memory after processing each file
    del df_temp
    
# Show first few rows
print(obs.head())

In [ ]:
channel = 'glycans_channel_108'

obs = obs.sort_values(by=channel, ascending=True)
sns.scatterplot(data=obs, x='x', y='y', hue=channel, palette='viridis', s=.45)

In [ ]:
obs.fillna(0, inplace=True)  # Fill NaN values with 0

In [ ]:
### perform normalization using z-score
from sklearn.preprocessing import StandardScaler

coordinates = obs[['x', 'y']].values
intensities = obs.drop(columns=['x', 'y']).values

scaler = StandardScaler()
intensities_scaled = scaler.fit_transform(intensities)

# Create a new DataFrame with scaled intensities
scaled_obs = pd.DataFrame(intensities_scaled, columns=obs.columns[2:], index=obs.index)
scaled_obs[['x', 'y']] = obs[['x', 'y']].values

### perform PCA on the scaled intensities and UMAP for visualization
from sklearn.decomposition import PCA
from umap import UMAP   
pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_obs.drop(columns=['x', 'y']).values)


In [ ]:

umap_model = UMAP(n_neighbors=50, min_dist=0.1, metric='euclidean')
umap_result = umap_model.fit_transform(pca_result)  

# Create a DataFrame for UMAP results
umap_df = pd.DataFrame(umap_result, columns=['UMAP1', 'UMAP2'])
umap_df[['x', 'y']] = scaled_obs[['x', 'y']].values


#plt.savefig(dst_folder / 'umap_projection.png', dpi=300)

In [ ]:
final_df = pd.concat([umap_df, obs.drop(columns=['x', 'y'])], axis=1)

In [ ]:
final_df

In [ ]:
# Plot UMAP results
plt.figure(figsize=(10, 8))
sns.scatterplot(data=final_df, x='UMAP1', y='UMAP2', hue=channel, palette='viridis', s=10)
plt.title('UMAP Projection of Scaled Intensities')
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()